**生成式 AI 使用声明**：就本作业而言，使用生成式 AI 工具须遵守与合作相同的政策。与与其他合作者一样，每位学生必须独立于交互输出之外写下解决方案，且提交的作业应注明合作的性质。使用生成式 AI 工具实质性完成作业的某些部分不符合作业的精神，将违反[荣誉守则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 这将把你的 Google Drive 挂载到 Colab 虚拟机。
from google.colab import drive
drive.mount('/content/drive')

# TODO: 输入你在 Drive 中保存了解压后作业文件夹的路径，
# 例如 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] 请输入文件夹名称。"

# 现在我们已经挂载了你的 Drive，这确保
# Colab 虚拟机的 Python 解释器可以从中
# 加载 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 如果 CIFAR-10 数据集不存在，这将下载它到你的 Drive。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Dropout
Dropout [1] 是一种通过在正向传播期间随机将某些输出激活设置为零来正则化神经网络的技术。在这个练习中，你将实现一个 dropout 层并修改你的全连接网络以选择性地使用 dropout。

[1] [Geoffrey E. Hinton et al, "Improving neural networks by preventing co-adaptation of feature detectors", arXiv 2012](https://arxiv.org/abs/1207.0580)

In [ ]:
# 设置单元格。
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # 设置默认绘图大小。
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """返回相对误差。"""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载（预处理后的）CIFAR-10 数据。
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Dropout：前向传播
在文件 `cs231n/layers.py` 中，实现 dropout 的前向传播。由于 dropout 在训练和测试期间表现不同，请确保为两种模式都实现该操作。

完成后，运行下面的单元格来测试你的实现。

In [ ]:
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})

    print('使用 p = ', p, '运行测试')
    print('输入均值: ', x.mean())
    print('训练时输出均值: ', out.mean())
    print('测试时输出均值: ', out_test.mean())
    print('训练时输出被置为零的比例: ', (out == 0).mean())
    print('测试时输出被置为零的比例: ', (out_test == 0).mean())
    print()

# Dropout：反向传播
在文件 `cs231n/layers.py` 中，实现 dropout 的反向传播。完成后，运行以下单元格用数值方法梯度检查你的实现。

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)

dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)

# 误差应在 e-10 或更小。
print('dx 相对误差: ', rel_error(dx, dx_num))

## 思考题 1：
如果我们在 dropout 层中不将逆 dropout 传递的值除以 `p`，会发生什么？为什么会这样？

## 回答：
[在此填写]

# 使用 Dropout 的全连接网络
在文件 `cs231n/classifiers/fc_net.py` 中，修改你的实现以使用 dropout。具体来说，如果网络构造函数收到的 `dropout_keep_ratio` 参数值不是 1，则网络应在每个 ReLU 非线性之后立即添加一个 dropout 层。完成后，运行以下代码用数值方法梯度检查你的实现。

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for dropout_keep_ratio in [1, 0.75, 0.5]:
    print('使用 dropout = ', dropout_keep_ratio, '运行检查')
    model = FullyConnectedNet(
        [H1, H2],
        input_dim=D,
        num_classes=C,
        weight_scale=5e-2,
        dtype=np.float64,
        dropout_keep_ratio=dropout_keep_ratio,
        seed=123
    )

    loss, grads = model.loss(X, y)
    print('初始损失: ', loss)

    # 相对误差应在 e-6 或更小。
    # 注意，对于 dropout_keep_ratio=1，W2 误差在 e-5 数量级是可以的。
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print('%s 相对误差: %.2e' % (name, rel_error(grad_num, grads[name])))
    print()

# 正则化实验
作为实验，我们将在 500 个训练样本上训练一对两层网络：一个不使用 dropout，另一个使用保留概率 0.25。然后我们将可视化两个网络随时间的训练和验证准确率。

In [ ]:
# 训练两个相同的网络，一个使用 dropout，一个不使用。
np.random.seed(231)
num_train = 500
small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

solvers = {}
dropout_choices = [1, 0.25]
for dropout_keep_ratio in dropout_choices:
    model = FullyConnectedNet(
        [500],
        dropout_keep_ratio=dropout_keep_ratio
    )
    print(dropout_keep_ratio)

    solver = Solver(
        model,
        small_data,
        num_epochs=25,
        batch_size=100,
        update_rule='adam',
        optim_config={'learning_rate': 5e-4,},
        verbose=True,
        print_every=100
    )
    solver.train()
    solvers[dropout_keep_ratio] = solver
    print()

In [ ]:
# 绘制两个模型的训练和验证准确率。
train_accs = []
val_accs = []
for dropout_keep_ratio in dropout_choices:
    solver = solvers[dropout_keep_ratio]
    train_accs.append(solver.train_acc_history[-1])
    val_accs.append(solver.val_acc_history[-1])

plt.subplot(3, 1, 1)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].train_acc_history, 'o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('训练准确率')
plt.xlabel('轮次')
plt.ylabel('准确率')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].val_acc_history, 'o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('验证准确率')
plt.xlabel('轮次')
plt.ylabel('准确率')
plt.legend(ncol=2, loc='lower right')

plt.gcf().set_size_inches(15, 15)
plt.show()

## 思考题 2：
比较使用和不使用 dropout 的验证和训练准确率——你的结果对 dropout 作为正则化器有何启示？

## 回答：
[在此填写]